# LoViF 2026 - StableCodec Native Pretrained Submission

This notebook replaces the blurry hand-built proxy with a **no-training pretrained native low-bitrate codec**. It uses StableCodec checkpoints trained around the LoViF bitrate regime, then wraps the produced files into the Codabench development format.

Default behavior:
- Use `stablecodec_ft24.pkl` first because the StableCodec README reports it near 0.008 bpp on Kodak.
- If the measured LoViF validation BPP is above 0.008 or files are missing, fall back to `stablecodec_ft32.pkl`.
- Do **not** use StableCodec's source-image color fix by default, because it uses original-image color statistics after decompression and those stats are not written into the bitstream.
- Produce `/kaggle/working/lovif_stablecodec_submission.zip`.

No training or fine-tuning is performed.


In [1]:

from __future__ import annotations

import os
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path

from PIL import Image

# =========================
# User Config
# =========================
BPP_LIMIT = 0.008
SEED = 123
STOP_AFTER_FIRST_VALID = True
RUN_LOCAL_PYIQA_EVAL = False  # Very slow. Submit to Codabench first; enable only for local diagnostics.

# If you upload StableCodec/SD/checkpoints as Kaggle datasets, the notebook will auto-detect them.
# Otherwise it downloads from GitHub, Hugging Face, and Google Drive.
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
LOCAL_WORKING = Path.cwd()
IS_KAGGLE = KAGGLE_WORKING.exists()
WORK = KAGGLE_WORKING if IS_KAGGLE else LOCAL_WORKING / 'stablecodec_work'
WORK.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORK / 'StableCodec'
MODEL_DIR = WORK / 'models'
CKPT_DIR = MODEL_DIR / 'stablecodec_ckpts'
SD_TURBO_DIR = MODEL_DIR / 'sd-turbo'
OUT_ROOT = WORK / 'stablecodec_candidates'
FINAL_ROOT = WORK / 'lovif_stablecodec_submission'
FINAL_ZIP = WORK / 'lovif_stablecodec_submission.zip'

# LoViF mounted dataset path can vary. This auto-discovers dataset_val.
DATASET_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/tonyironman099/lovif-2026-image-compression'),
    Path('/kaggle/input/lovif-2026-image-compression'),
    Path('/kaggle/input'),
    Path('filtered_dataset'),
]

# StableCodec Google Drive file IDs, listed from the public folder in the README.
GDRIVE_IDS = {
    'elic_official.pth': '1jUfYJdZd0-bYUsoOUWwEpI5t1MZYP3AP',
    'stablecodec_ft24.pkl': '1grLxmLuth4ydXhghZwa9wxGhjnRGzXim',
    'stablecodec_ft32.pkl': '1quyX_-g4B05DQrMb5bGlonaFrOlLyd8i',
}

@dataclass
class Candidate:
    name: str
    checkpoint: str
    color_fix: bool = False

CANDIDATES = [
    Candidate('stablecodec_ft24_clean', 'stablecodec_ft24.pkl', False),
    Candidate('stablecodec_ft32_clean', 'stablecodec_ft32.pkl', False),
]


def run(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return result.returncode


def find_dataset_val() -> Path:
    for root in DATASET_ROOT_CANDIDATES:
        if not root.exists():
            continue
        direct = root / 'dataset_val'
        if direct.is_dir() and list(direct.glob('*.png')):
            return direct
        nested = list(root.rglob('dataset_val'))
        for p in nested:
            if p.is_dir() and list(p.glob('*.png')):
                return p
    raise FileNotFoundError('Could not find dataset_val. Check the Kaggle dataset mount path.')

INPUT_DIR = find_dataset_val()
input_paths = sorted(INPUT_DIR.glob('*.png'))
print(f'Using INPUT_DIR={INPUT_DIR}')
print(f'Found {len(input_paths)} validation images')

if not input_paths:
    raise RuntimeError('No PNG images found in dataset_val')

# =========================
# Install Runtime Dependencies
# =========================
# Do not reinstall torch on Kaggle; use the runtime-provided CUDA build.
packages = [
    'gdown',
    'huggingface_hub==0.25.0',
    'diffusers==0.25.1',
    'transformers==4.46.3',
    'accelerate==1.9.0',
    'peft',
    'omegaconf',
    'einops>=0.6.1',
    'timm==1.0.22',
    'compressai==1.2.6',
    'open-clip-torch>=2.20.0',
    'lpips',
]
run([sys.executable, '-m', 'pip', 'install', '-q', *packages])

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('StableCodec needs a CUDA GPU. On Kaggle, enable GPU accelerator.')

# =========================
# Clone or Locate StableCodec
# =========================
def find_existing_dir_with(marker_name: str):
    roots = [KAGGLE_INPUT, LOCAL_WORKING, WORK]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob(marker_name):
            if p.is_file():
                return p.parent
    return None

if not (REPO_DIR / 'src' / 'compress.py').exists():
    existing = find_existing_dir_with('compress.sh')
    if existing and (existing / 'src' / 'compress.py').exists():
        print(f'Copying existing StableCodec repo from {existing}')
        shutil.copytree(existing, REPO_DIR, dirs_exist_ok=True)
    else:
        run(['git', 'clone', '--depth', '1', 'https://github.com/LuizScarlet/StableCodec.git', REPO_DIR])
else:
    print(f'Using existing repo at {REPO_DIR}')

# =========================
# Download or Locate Pretrained Models
# =========================
def find_existing_file(filename: str):
    search_roots = [CKPT_DIR, MODEL_DIR, KAGGLE_INPUT, LOCAL_WORKING]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None

CKPT_DIR.mkdir(parents=True, exist_ok=True)

def ensure_gdrive_file(filename: str) -> Path:
    found = find_existing_file(filename)
    if found:
        print(f'Found {filename}: {found}')
        return found
    import gdown
    file_id = GDRIVE_IDS[filename]
    out = CKPT_DIR / filename
    print(f'Downloading {filename} from Google Drive...')
    gdown.download(url=f'https://drive.google.com/uc?id={file_id}', output=str(out), quiet=False, use_cookies=False)
    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError(f'Failed to download {filename}')
    return out

elic_path = ensure_gdrive_file('elic_official.pth')
ckpt_paths = {c.checkpoint: ensure_gdrive_file(c.checkpoint) for c in CANDIDATES}

# SD-Turbo can be provided as a Kaggle dataset or downloaded from Hugging Face.
def find_sd_turbo_dir():
    for root in [SD_TURBO_DIR, KAGGLE_INPUT, LOCAL_WORKING]:
        if not root.exists():
            continue
        for p in root.rglob('model_index.json'):
            if 'sd-turbo' in str(p.parent).lower() or (p.parent / 'unet').exists():
                return p.parent
    return None

sd_found = find_sd_turbo_dir()
if sd_found:
    sd_path = sd_found
    print(f'Found SD-Turbo at {sd_path}')
else:
    from huggingface_hub import snapshot_download
    print('Downloading SD-Turbo from Hugging Face...')
    sd_path = Path(snapshot_download(
        repo_id='stabilityai/sd-turbo',
        local_dir=str(SD_TURBO_DIR),
        local_dir_use_symlinks=False,
    ))
print(f'SD path: {sd_path}')

# =========================
# Run StableCodec Candidate(s)
# =========================
def stablecodec_command(candidate: Candidate, rec_dir: Path, bin_dir: Path):
    cmd = [
        sys.executable, 'compress.py',
        '--sd_path', sd_path,
        '--elic_path', elic_path,
        '--img_path', INPUT_DIR,
        '--rec_path', rec_dir,
        '--bin_path', bin_dir,
        '--codec_path', ckpt_paths[candidate.checkpoint],
        '--seed', str(SEED),
    ]
    if candidate.color_fix:
        cmd.append('--color_fix')
    return cmd


def count_pixels(path: Path) -> int:
    with Image.open(path) as img:
        w, h = img.size
    return w * h


def summarize_candidate(bin_dir: Path):
    total_bytes = 0
    total_pixels = 0
    missing = []
    per_image = []
    for src in input_paths:
        bit = bin_dir / src.stem
        if not bit.exists():
            missing.append(src.name)
            continue
        pixels = count_pixels(src)
        size = bit.stat().st_size
        total_bytes += size
        total_pixels += pixels
        per_image.append((src.name, size, 8.0 * size / pixels))
    avg_bpp = 8.0 * total_bytes / total_pixels if total_pixels else 999.0
    return {'avg_bpp': avg_bpp, 'missing': missing, 'per_image': per_image, 'total_bytes': total_bytes, 'total_pixels': total_pixels}

candidate_results = []
selected = None
OUT_ROOT.mkdir(parents=True, exist_ok=True)

for candidate in CANDIDATES:
    cand_root = OUT_ROOT / candidate.name
    rec_dir = cand_root / 'rec'
    bin_dir = cand_root / 'bin'
    if cand_root.exists():
        shutil.rmtree(cand_root)
    rec_dir.mkdir(parents=True, exist_ok=True)
    bin_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n=== Running {candidate.name} ({candidate.checkpoint}) ===')
    start = time.perf_counter()
    run(stablecodec_command(candidate, rec_dir, bin_dir), cwd=REPO_DIR / 'src')
    elapsed = time.perf_counter() - start
    summary = summarize_candidate(bin_dir)
    summary.update({'name': candidate.name, 'checkpoint': candidate.checkpoint, 'elapsed_s': elapsed, 'runtime_per_image_s': elapsed / len(input_paths)})
    candidate_results.append(summary)
    print(f"Candidate {candidate.name}: avg_bpp={summary['avg_bpp']:.6f}, missing={len(summary['missing'])}, runtime/img={summary['runtime_per_image_s']:.2f}s")

    if not summary['missing'] and summary['avg_bpp'] <= BPP_LIMIT:
        selected = (candidate, cand_root, summary)
        if STOP_AFTER_FIRST_VALID:
            break

if selected is None:
    print('No valid candidate selected. Candidate summaries:')
    for r in candidate_results:
        print(r['name'], 'bpp=', r['avg_bpp'], 'missing=', len(r['missing']))
    raise RuntimeError('No StableCodec candidate satisfied the BPP/files constraint.')

candidate, cand_root, summary = selected
print(f"\nSelected {candidate.name}: BPP={summary['avg_bpp']:.6f}")

# =========================
# Wrap into LoViF Submission Format
# =========================
def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

reset_dir(FINAL_ROOT)
rec_out = FINAL_ROOT / 'reconstructed'
bit_out = FINAL_ROOT / 'bitstream'
rec_out.mkdir()
bit_out.mkdir()

rec_src = cand_root / 'rec'
bin_src = cand_root / 'bin'
copy_errors = []
for src in input_paths:
    rec = rec_src / src.name
    bit = bin_src / src.stem
    if not rec.exists():
        copy_errors.append(f'missing reconstruction {src.name}')
        continue
    if not bit.exists():
        copy_errors.append(f'missing bitstream {src.stem}')
        continue
    shutil.copy2(rec, rec_out / src.name)
    shutil.copy2(bit, bit_out / f'{src.stem}.bin')

if copy_errors:
    raise RuntimeError('\n'.join(copy_errors[:20]))

readme = f'''runtime per image [s] : {summary['runtime_per_image_s']:.4f}
CPU[1] / GPU[0] : 0
Extra Data [1] / No Extra Data [0] : 1
Other description: No-training pretrained StableCodec submission. Used {candidate.checkpoint}, SD-Turbo, StableCodec LoRA/latent-codec checkpoint, and ELIC auxiliary encoder. StableCodec bitstreams were wrapped with .bin extension for LoViF format. Color fix: {candidate.color_fix}. Average BPP measured locally: {summary['avg_bpp']:.6f}.
'''
(FINAL_ROOT / 'readme.txt').write_text(readme, encoding='utf-8')

# Validate exact filenames, resolution, and final BPP from wrapped .bin files.
def validate_submission(root: Path):
    total_bytes = 0
    total_pixels = 0
    errors = []
    for src in input_paths:
        rec = root / 'reconstructed' / src.name
        bit = root / 'bitstream' / f'{src.stem}.bin'
        if not rec.exists():
            errors.append(f'missing rec {src.name}')
            continue
        if not bit.exists():
            errors.append(f'missing bin {src.stem}.bin')
            continue
        with Image.open(src) as gt, Image.open(rec) as rr:
            if gt.size != rr.size:
                errors.append(f'size mismatch {src.name}: gt={gt.size}, rec={rr.size}')
            pixels = gt.size[0] * gt.size[1]
        total_bytes += bit.stat().st_size
        total_pixels += pixels
    avg_bpp = 8.0 * total_bytes / total_pixels
    return errors, avg_bpp, total_bytes, total_pixels

errors, final_bpp, final_bytes, final_pixels = validate_submission(FINAL_ROOT)
print('\nWrapped validation')
print('errors:', len(errors))
print('avg_bpp:', final_bpp)
print('total bitstream bytes:', final_bytes)
if errors:
    print('\n'.join(errors[:20]))
    raise RuntimeError('Invalid wrapped submission')
if final_bpp > BPP_LIMIT:
    raise RuntimeError(f'BPP too high: {final_bpp:.6f} > {BPP_LIMIT}')

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()
with zipfile.ZipFile(FINAL_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted((FINAL_ROOT / 'reconstructed').glob('*.png')):
        zf.write(p, f'reconstructed/{p.name}')
    for p in sorted((FINAL_ROOT / 'bitstream').glob('*.bin')):
        zf.write(p, f'bitstream/{p.name}')
    zf.write(FINAL_ROOT / 'readme.txt', 'readme.txt')

print(f'\nCreated submission: {FINAL_ZIP}')
print(f'Final BPP: {final_bpp:.6f} / {BPP_LIMIT}')
print(f'Selected checkpoint: {candidate.checkpoint}')
print(f'Readme:\n{readme}')

# Optional local metric evaluation. This is slow and the Codabench LPIPS/DISTS profile has quirks.
if RUN_LOCAL_PYIQA_EVAL:
    eval_script = Path('scripts/evaluate_submission.py')
    if eval_script.exists():
        run([sys.executable, eval_script, INPUT_DIR, FINAL_ZIP, '--install-missing'])
    else:
        print('Local evaluator script not found; skipping pyiqa evaluation.')


Using INPUT_DIR=/kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_val
Found 100 validation images

$ /usr/bin/python3 -m pip install -q gdown huggingface_hub==0.25.0 diffusers==0.25.1 transformers==4.46.3 accelerate==1.9.0 peft omegaconf einops>=0.6.1 timm==1.0.22 compressai==1.2.6 open-clip-torch>=2.20.0 lpips
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 773.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 323.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

torch: 2.10.0+cu128 cuda: True

$ git clone --depth 1 https://github.com/LuizScarlet/StableCodec.git /kaggle/working/StableCodec


Cloning into '/kaggle/working/StableCodec'...


Downloading...
From (original): https://drive.google.com/uc?id=1jUfYJdZd0-bYUsoOUWwEpI5t1MZYP3AP
From (redirected): https://drive.google.com/uc?id=1jUfYJdZd0-bYUsoOUWwEpI5t1MZYP3AP&confirm=t&uuid=3602b146-d381-4b48-b507-a9990f45bfe8
To: /kaggle/working/models/stablecodec_ckpts/elic_official.pth
100%|██████████| 163M/163M [00:02<00:00, 62.0MB/s]


...

[Tiled VAE]: Done in 20.624s, max VRAM alloc 7256.419 MB
[BPP] 0.006670511341791619
[Processing] /kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_val/0062.png
[Tiled VAE]: input_size: torch.Size([1, 3, 1792, 1280]), tile_size: 1024, padding: 32
[Tiled VAE]: split to 2x2 = 4 tiles. Optimal tile size 608x864, original tile size 1024x1024


[Tiled VAE]: Executing Encoder Task Queue: 100%|██████████| 364/364 [00:22<00:00, 15.93it/s] 


[Tiled VAE]: Done in 23.580s, max VRAM alloc 12739.627 MB
[Tiled Latent]: the input latent is 224x160, need to tiled
[Tiled VAE]: input_size: torch.Size([1, 4, 224, 160]), tile_size: 160, padding: 11
[Tiled VAE]: split to 2x1 = 2 tiles. Optimal tile size 160x128, original tile size 160x160


[Tiled VAE]: Executing Decoder Task Queue: 100%|██████████| 246/246 [00:52<00:00,  4.71it/s]


[Tiled VAE]: Done in 53.019s, max VRAM alloc 8844.880 MB
[BPP] 0.00861664730837603

[Average BPP] 0.006885424769565746
Candidate stablecodec_ft24_clean: avg_bpp=0.006703, missing=0, runtime/img=72.76s

Selected stablecodec_ft24_clean: BPP=0.006703

Wrapped validation
errors: 0
avg_bpp: 0.006702759231984906
total bitstream bytes: 209128

Created submission: /kaggle/working/lovif_stablecodec_submission.zip
Final BPP: 0.006703 / 0.008
Selected checkpoint: stablecodec_ft24.pkl
Readme:
runtime per image [s] : 72.7631
CPU[1] / GPU[0] : 0
Extra Data [1] / No Extra Data [0] : 1
Other description: No-training pretrained StableCodec submission. Used stablecodec_ft24.pkl, SD-Turbo, StableCodec LoRA/latent-codec checkpoint, and ELIC auxiliary encoder. StableCodec bitstreams were wrapped with .bin extension for LoViF format. Color fix: False. Average BPP measured locally: 0.006703.

